# RAG Customer Support Pipeline

This notebook implements the Retrieval-Augmented Generation (RAG)
component of the e-commerce customer support chatbot.

The Bitext customer-support dataset is used as the knowledge base.

The RAG pipeline consists of:

1. Customer question
2. Sentence Transformer embedding
3. Chroma vector database retrieval
4. Top relevant customer-support examples
5. Groq LLM generation
6. Grounded final response

The retrieved Bitext responses provide the context used by the language
model. If the retrieved context does not contain enough information,
the chatbot should avoid guessing and offer human support instead.

In [127]:
from datasets import load_dataset
import pandas as pd
import numpy as np

from pathlib import Path
from sentence_transformers import SentenceTransformer

import chromadb

from dotenv import load_dotenv
import os

from groq import Groq

In [128]:
current_path = Path.cwd()

if current_path.name == "Jupyter notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

vector_db_dir = project_root / "vector_db"

vector_db_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", project_root)
print("Vector database:", vector_db_dir)

Project root: e:\iti\ass10
Vector database: e:\iti\ass10\vector_db


In [129]:
dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)

df = dataset["train"].to_pandas()

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Shape: (26872, 5)
Columns: ['flags', 'instruction', 'category', 'intent', 'response']


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [130]:
rag_df = df[
    [
        "instruction",
        "response",
        "intent",
        "category"
    ]
].copy()

print(rag_df.isnull().sum())

instruction    0
response       0
intent         0
category       0
dtype: int64


In [131]:
rag_df = (
    rag_df
    .dropna(
        subset=[
            "instruction",
            "response"
        ]
    )
    .drop_duplicates(
        subset=["instruction"]
    )
    .reset_index(drop=True)
)

print(
    "Knowledge base size:",
    len(rag_df)
)

rag_df.head()

Knowledge base size: 24635


,instruction,response,intent,category
0,question about cancelling order {{Order Number}},I've understood you have a question regarding ...,cancel_order,ORDER
1,i have a question about cancelling oorder {{Or...,I've been informed that you have a question ab...,cancel_order,ORDER
2,i need help cancelling puchase {{Order Number}},I can sense that you're seeking assistance wit...,cancel_order,ORDER
3,I need to cancel purchase {{Order Number}},I understood that you need assistance with can...,cancel_order,ORDER
4,"I cannot afford this order, cancel purchase {{...",I'm sensitive to the fact that you're facing f...,cancel_order,ORDER


In [132]:
embedding_model_name = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

embedding_model = SentenceTransformer(
    embedding_model_name
)

print(
    "Embedding model:",
    embedding_model_name
)

print(
    "Embedding dimension:",
    embedding_model.get_embedding_dimension()
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Embedding dimension: 384


In [133]:
sample_embedding = embedding_model.encode(
    "Where is my order?"
)

print(
    "Embedding shape:",
    sample_embedding.shape
)

print(
    sample_embedding[:10]
)

Embedding shape: (384,)
[ 0.05841671 -0.3444181   0.28886047 -0.02290464 -0.25380555 -0.02673797
  0.10802222  0.08271708  0.15906408  0.08218485]


In [134]:
chroma_client = chromadb.PersistentClient(
    path=str(vector_db_dir)
)

collection = chroma_client.get_or_create_collection(
    name="customer_support",
    metadata={
        "hnsw:space": "cosine"
    }
)

print(
    "Existing documents:",
    collection.count()
)

Existing documents: 24635


In [135]:
documents = (
    rag_df["instruction"]
    .astype(str)
    .tolist()
)

responses = (
    rag_df["response"]
    .astype(str)
    .tolist()
)

intents = (
    rag_df["intent"]
    .astype(str)
    .tolist()
)

categories = (
    rag_df["category"]
    .astype(str)
    .tolist()
)

ids = [
    f"support_{i}"
    for i in range(len(rag_df))
]

print(
    "Documents prepared:",
    len(documents)
)

Documents prepared: 24635


In [136]:
BATCH_SIZE = 512

if collection.count() == 0:

    for start in range(
        0,
        len(documents),
        BATCH_SIZE
    ):

        end = min(
            start + BATCH_SIZE,
            len(documents)
        )

        batch_documents = documents[
            start:end
        ]

        batch_embeddings = (
            embedding_model.encode(
                batch_documents,
                show_progress_bar=False,
                normalize_embeddings=True
            )
        )

        batch_metadatas = []

        for i in range(start, end):

            batch_metadatas.append({
                "response": responses[i],
                "intent": intents[i],
                "category": categories[i]
            })

        collection.add(
            ids=ids[start:end],
            documents=batch_documents,
            embeddings=batch_embeddings.tolist(),
            metadatas=batch_metadatas
        )

        print(
            f"Stored {end}/{len(documents)}"
        )

else:

    print(
        "Vector database already exists."
    )

Vector database already exists.


In [137]:
print(
    "Documents in Chroma:",
    collection.count()
)

Documents in Chroma: 24635


In [138]:
def retrieve_support(
    query,
    top_k=5
):

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=top_k
    )

    retrieved = []

    for i in range(
        len(results["documents"][0])
    ):

        retrieved.append({
            "instruction":
                results["documents"][0][i],

            "response":
                results["metadatas"][0][i][
                    "response"
                ],

            "intent":
                results["metadatas"][0][i][
                    "intent"
                ],

            "category":
                results["metadatas"][0][i][
                    "category"
                ],

            "distance":
                results["distances"][0][i]
        })

    return retrieved

In [139]:
query = "How can I track my order?"

retrieved_results = retrieve_support(
    query,
    top_k=5
)

for i, item in enumerate(
    retrieved_results,
    start=1
):

    print(
        f"Result {i}"
    )

    print(
        "Instruction:",
        item["instruction"]
    )

    print(
        "Intent:",
        item["intent"]
    )

    print(
        "Category:",
        item["category"]
    )

    print(
        "Distance:",
        round(
            item["distance"],
            4
        )
    )

    print(
        "Response:",
        item["response"]
    )

    print("-" * 80)

Result 1
Instruction: what do I need to do to track order {{Order Number}}?
Intent: track_order
Category: ORDER
Distance: 0.1541
Response: Thank you for your question! To track the status of your order with the order number {{Order Number}}, you can follow these simple steps:

1. Visit our website and log into your account using your credentials.
2. Navigate to the "Order History" section, which should be easily accessible from the main menu.
3. Once you're on the "Order History" page, locate the specific order number ({{Order Number}}) you want to track.
4. Click on the order number, and you'll be redirected to a detailed order summary page.
5. On the order summary page, you should find the current status of your order, along with any relevant tracking information such as shipping carrier and tracking number.
6. If you have any further questions or encounter any issues while tracking your order, don't hesitate to reach out to our customer support team, available {{Customer Support Hou

In [140]:
retrieval_tests = [
    "I want my money back",
    "Where is my package?",
    "How can I change my shipping address?",
    "I forgot my password",
    "My payment is not working"
]

for query in retrieval_tests:

    results = retrieve_support(
        query,
        top_k=3
    )

    print(
        f"\nQUERY: {query}"
    )

    for result in results:

        print(
            "→",
            result["intent"],
            "|",
            round(
                result["distance"],
                4
            )
        )


QUERY: I want my money back
→ get_refund | 0.1145
→ get_refund | 0.115
→ get_refund | 0.1167

QUERY: Where is my package?
→ delivery_period | 0.2339
→ delivery_period | 0.2368
→ delivery_period | 0.2538

QUERY: How can I change my shipping address?
→ change_shipping_address | 0.0001
→ change_shipping_address | 0.0193
→ change_shipping_address | 0.0222

QUERY: I forgot my password
→ recover_password | 0.0918
→ recover_password | 0.117
→ recover_password | 0.1224

QUERY: My payment is not working
→ payment_issue | 0.1227
→ payment_issue | 0.1263
→ payment_issue | 0.1276


In [141]:
load_dotenv(
    project_root / ".env"
)

groq_api_key = os.getenv(
    "GROQ_API_KEY"
)

if not groq_api_key:
    raise ValueError(
        "GROQ_API_KEY was not found in the .env file."
    )

client = Groq(
    api_key=groq_api_key
)

print("Groq client initialized successfully.")

Groq client initialized successfully.


In [142]:
def translate_query_to_english(
    query,
    language
):
    if language.lower() == "english":
        return query

    completion = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": (
                    "Translate the customer message into clear English. "
                    "Preserve the original e-commerce support intent. "
                    "Return only the English translation."
                )
            },
            {
                "role": "user",
                "content": query
            }
        ],
        temperature=0.0,
        max_completion_tokens=100
    )

    return (
        completion
        .choices[0]
        .message
        .content
        .strip()
    )

In [143]:
def build_context(
    retrieved_results
):

    context_parts = []

    for i, item in enumerate(
        retrieved_results,
        start=1
    ):

        context_parts.append(
            f"""
Support Example {i}

Customer Question:
{item["instruction"]}

Support Response:
{item["response"]}

Intent:
{item["intent"]}

Category:
{item["category"]}
"""
        )

    return "\n".join(
        context_parts
    )

In [144]:
test_results = retrieve_support(
    "How can I track my order?",
    top_k=3
)

test_context = build_context(
    test_results
)

print(test_context)


Support Example 1

Customer Question:
what do I need to do to track order {{Order Number}}?

Support Response:
Thank you for your question! To track the status of your order with the order number {{Order Number}}, you can follow these simple steps:

1. Visit our website and log into your account using your credentials.
2. Navigate to the "Order History" section, which should be easily accessible from the main menu.
3. Once you're on the "Order History" page, locate the specific order number ({{Order Number}}) you want to track.
4. Click on the order number, and you'll be redirected to a detailed order summary page.
5. On the order summary page, you should find the current status of your order, along with any relevant tracking information such as shipping carrier and tracking number.
6. If you have any further questions or encounter any issues while tracking your order, don't hesitate to reach out to our customer support team, available {{Customer Support Hours}} at {{Customer Support 

In [145]:
MAX_DISTANCE = 0.55

def filter_relevant_results(
    results,
    max_distance=MAX_DISTANCE
):

    return [
        item
        for item in results
        if item["distance"] <= max_distance
    ]

In [146]:
def build_system_prompt(
    language,
    sentiment,
    intent
):

    empathy_instruction = ""

    if sentiment == "negative":
        empathy_instruction = (
            "The customer appears frustrated or upset. "
            "Acknowledge their frustration briefly and respond empathetically."
        )

    return f"""
You are an e-commerce customer support assistant.

Customer information:
- Detected language: {language}
- Sentiment: {sentiment}
- Intent: {intent}

Rules:

1. Answer using ONLY the provided customer-support context.
2. Do not invent policies, order details, refund amounts, dates, tracking information, or customer information.
3. If the context does not contain enough information, clearly say that you do not have enough information and offer escalation to a human support agent.
4. Respond in the customer's detected language.
5. Keep the answer clear, helpful, and concise.
6. Do not mention that you are using a dataset, retrieval system, vector database, or RAG.
7. Preserve useful placeholders such as order number or website information when needed.
8. {empathy_instruction}
"""

In [147]:
def generate_rag_answer(
    query,
    language="English",
    sentiment="neutral",
    intent="unknown",
    top_k=5
):

    retrieval_query = translate_query_to_english(
        query,
        language
    )

    retrieved_results = retrieve_support(
        retrieval_query,
        top_k=top_k
    )

    relevant_results = filter_relevant_results(
        retrieved_results
    )

    if relevant_results:

        context = build_context(
            relevant_results
        )

    else:

        context = (
            "No sufficiently relevant "
            "customer-support context was found."
        )

    system_prompt = build_system_prompt(
        language=language,
        sentiment=sentiment,
        intent=intent
    )

    user_prompt = f"""
Customer message:
{query}

Retrieved customer-support context:

{context}

Provide the final customer-support response.
"""

    completion = client.chat.completions.create(
        model="openai/gpt-oss-20b",

        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],

        temperature=0.2,
        max_completion_tokens=500
    )

    answer = (
        completion
        .choices[0]
        .message
        .content
    )

    return {
        "answer": answer,
        "retrieval_query": retrieval_query,
        "retrieved": relevant_results
    }

In [148]:
result = generate_rag_answer(
    query="How can I track my order?",
    language="English",
    sentiment="neutral",
    intent="order_status"
)

print(
    result["answer"]
)

Sure! To see the current status of your order, follow these steps:

1. **Log in** to your account on our website.  
2. Go to the **“Order History”** (or “My Orders”) section.  
3. Find the order you want to track and click on its number.  
4. On the order detail page, you’ll see the latest status and any tracking number or carrier information.

If you need help locating your order or have the order number handy, just let me know and I can walk you through it.


In [149]:
result = generate_rag_answer(
    query=(
        "I am extremely angry. "
        "My payment keeps failing!"
    ),
    language="English",
    sentiment="negative",
    intent="billing_and_refunds"
)

print(
    result["answer"]
)

I’m really sorry you’re having this frustrating experience.  
Could you please share any error messages or codes you see when the payment fails, and the type of card or payment method you’re using? That will help us pinpoint the issue.  

If you’d prefer to speak with a live agent right away, I can connect you to one.


In [150]:
query = (
    "What is the capital of France?"
)

results = retrieve_support(
    query,
    top_k=5
)

for result in results:

    print(
        result["intent"],
        round(
            result["distance"],
            4
        )
    )

delivery_options 0.7056
delivery_options 0.7096
delivery_options 0.7245
delivery_options 0.7295
delivery_options 0.7341


In [151]:
result = generate_rag_answer(
    query="What is the capital of France?",
    language="English",
    sentiment="neutral",
    intent="out_of_scope"
)

print(
    result["answer"]
)

I’m sorry, but I don’t have enough information to answer that. I can connect you with a human support agent who can help.


In [152]:
result = generate_rag_answer(
    query="أريد أن أعرف أين طلبي",
    language="Arabic",
    sentiment="neutral",
    intent="order_status"
)

print(
    "Retrieval query:",
    result["retrieval_query"]
)

print("\nRetrieved results:")

for item in result["retrieved"]:
    print(
        item["intent"],
        "|",
        round(item["distance"], 4)
    )

print("\nFinal answer:")
print(
    result["answer"]
)

Retrieval query: I want to know where my order is.

Retrieved results:
delivery_period | 0.1682
delivery_period | 0.1728
delivery_period | 0.1794
delivery_period | 0.1843
delivery_period | 0.1938

Final answer:
نحن نقدر رغبتك في معرفة مكان طلبك. هل يمكنك تزويدنا برقم الطلب أو رقم التتبع؟ مع هذه المعلومات، يمكننا التحقق من حالة الشحن وتزويدك بالتحديثات الأخيرة.


## Conclusion

The RAG module was implemented using the Bitext customer-support dataset as
the knowledge base, Sentence Transformers for semantic embeddings, Chroma
for persistent vector retrieval, and Groq for response generation.

After duplicate instructions were removed, the knowledge base contained
24,635 customer-support examples.

Customer questions are embedded and matched against the knowledge base
using cosine similarity. Retrieved support responses are supplied to the
LLM as grounding context.

A maximum cosine distance threshold of 0.55 is used to reject weak retrieval
results. This prevented unrelated questions, such as general-knowledge
questions, from being answered using the LLM's internal knowledge.

For non-English messages, the customer query is translated into English
before retrieval because the Bitext knowledge base is primarily English.
The original message and detected language are preserved so the final answer
can still be generated in the customer's language.

The prompt instructs the LLM to use only retrieved support information,
avoid inventing unsupported details, acknowledge negative customer sentiment,
and offer human escalation when sufficient context is unavailable.

The RAG pipeline was successfully tested with order tracking, payment issues,
out-of-scope questions, and Arabic customer-support queries.